# Dim_Exploração_scd & Dim_Exp_Geografia

In [ ]:
#Parameter 

run_id = ""

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print(f"{'='*80}")
print("CONSTRUÇÃO DA DIMENSÃO: gld.dim_exploracao_historico (SCD TIPO 2 - COMPRIMIDA)")
print(f"{'='*80}\n")


# 1. CARREGAR TABELAS SILVER
df_des = spark.read.table("slv.des")
df_siss = spark.read.table("slv.exploracoes_siss")


# 2. COMPRESSÃO DE HISTÓRICO DES (Algoritmo: Gaps and Islands)
# Ordenar o histórico por marca e data
win_asc = Window.partitionBy("marca").orderBy("data_inicio")

# Apanhar os valores da linha anterior
df_compress = df_des.withColumn("prev_sistema", F.lag("sistema_exploracao_origem").over(win_asc)) \
                    .withColumn("prev_tipo", F.lag("tipo_exploracao").over(win_asc)) \
                    .withColumn("prev_cn", F.lag("cn_valor").over(win_asc))

# Detetar mudança de estado (1 se mudou algo, 0 se manteve exatamente igual)
df_compress = df_compress.withColumn(
    "is_changed",
    F.when(
        (~F.col("sistema_exploracao_origem").eqNullSafe(F.col("prev_sistema"))) |
        (~F.col("tipo_exploracao").eqNullSafe(F.col("prev_tipo"))) |
        (~F.col("cn_valor").eqNullSafe(F.col("prev_cn"))) |
        F.col("prev_sistema").isNull(), 
        1
    ).otherwise(0)
)

# Criar ID do grupo consecutivo (Soma cumulativa de mudanças)
df_compress = df_compress.withColumn("island_id", F.sum("is_changed").over(win_asc))

# Janelas para agrupar a "ilha"
win_island = Window.partitionBy("marca", "island_id")
win_island_desc = Window.partitionBy("marca", "island_id").orderBy(F.desc("data_inicio"))

# Agregar datas e manter apenas a linha mais recente do grupo
df_des_compressed = df_compress.withColumn("data_inicio_agrupada", F.min("data_inicio").over(win_island)) \
                               .withColumn("data_fim_agrupada", F.max("data_fim").over(win_island)) \
                               .withColumn("nr_des_agrupados", F.count("*").over(win_island)) \
                               .withColumn("row_in_island", F.row_number().over(win_island_desc)) \
                               .filter(F.col("row_in_island") == 1)

# Limpar colunas temporárias e assumir as datas consolidadas
df_des = df_des_compressed.withColumn("data_inicio", F.col("data_inicio_agrupada")) \
                          .withColumn("data_fim", F.col("data_fim_agrupada")) \
                          .withColumn("estado", F.when(F.col("data_fim") == F.to_date(F.lit("9999-12-31")), "Ativa").otherwise("Inativa")) \
                          .drop("prev_sistema", "prev_tipo", "prev_cn", "is_changed", "island_id", "data_inicio_agrupada", "data_fim_agrupada", "row_in_island")


# 3. PREPARAÇÃO DO DES PARA O JOIN (Prefixar colunas)
for c in df_des.columns:
    if c != "marca":
        df_des = df_des.withColumnRenamed(c, f"des_{c}")


# 4. FULL OUTER JOIN & IDENTIFICAÇÃO DE REGISTOS
df_joined = df_des.join(df_siss, "marca", "full_outer")

df_joined = df_joined.withColumn("is_nova", F.col("des_data_inicio").isNull()) \
                     .withColumn("is_velha", F.col("des_data_inicio").isNotNull())

win_asc_final = Window.partitionBy("marca").orderBy("des_data_inicio")
df_joined = df_joined.withColumn(
    "des_is_first_record", 
    F.when(F.col("is_velha") & (F.row_number().over(win_asc_final) == 1), True).otherwise(False)
)


# 4.5. ATUALIZAÇÃO DA DSAVR COM BASE NO SVL (CORREÇÃO DE CARACTERES INCLUÍDA)
list_acores   = ["SDA Faial", "SDA Flores e Corvo", "SDA Graciosa", "SDA Pico", "SDA S. Jorge", "SDA S. Miguel", "SDA Sta. Maria", "SDA Terceira"]
list_madeira  = ["DRADR MADEIRA"]
list_alentejo = ["DAV Alentejo Central", "DAV Alentejo Litoral", "DAV Alto Alentejo", "DAV Baixo Alentejo", "NAV Aljustrel", "NAV Elvas", "NAV Odemira", "NAV Reguengos de Monsaraz", "NAV Santiago do Cacém", "NAV Serpa"]
list_centro   = ["DAV Aveiro", "DAV Castelo Branco", "DAV Coimbra", "DAV Guarda", "DAV Leiria", "DAV Viseu", "NAV Fundão", "NAV Gouveia"]
list_lvt      = ["DAV Oeste", "DAV Ribatejo", "DAV Setúbal", "NAV Caldas da Rainha", "NAV Montijo", "NAV Tomar"]
list_norte    = ["DAV Braga", "DAV Bragança", "DAV Chaves - Mirandela", "DAV Porto", "DAV Viana do Castelo", "DAV Vila Real - Douro Sul", "NAV Foz Côa", "NAV Macedo de Cavaleiros", "NAV Miranda do Douro", "NAV Mogadouro", "NAV Montalegre"]
list_algarve  = ["DSAVR Algarve"]

df_joined = df_joined.withColumn(
    "dsavr",
    F.when(F.col("svl").isin(list_acores), "Açores")
     .when(F.col("svl").isin(list_madeira), "Madeira")
     .when(F.col("svl").isin(list_alentejo), "Alentejo")
     .when(F.col("svl").isin(list_centro), "Centro")
     .when(F.col("svl").isin(list_lvt), "LVT")
     .when(F.col("svl").isin(list_norte), "Norte")
     .when(F.col("svl").isin(list_algarve), "Algarve")
     .otherwise(F.col("dsavr")) # Mantém o valor original se não encontrar correspondência
)


# 5. CRIAÇÃO DA CHAVE GEOGRÁFICA ÚNICA (SK_Geo_Exp)
geo_cols = ["freguesia", "concelho", "distrito", "svl", "dsavr"]

all_null_cond = F.lit(True)
for c in geo_cols:
    all_null_cond = all_null_cond & F.col(c).isNull()

df_joined = df_joined.withColumn(
    "SK_Geo_Exp", 
    F.when(all_null_cond, F.lit("GEO_UNDEFINED"))
     .otherwise(F.concat(F.lit("SK_GEO_"), F.md5(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("N/A")) for c in geo_cols]))))
)


# 6. APLICAÇÃO DAS REGRAS DE NEGÓCIO DA DIMENSÃO HISTÓRICO
df_gold = df_joined.select(
    "SK_Geo_Exp",
    "marca",
    F.when(F.col("latitude").isin(0.0, 0), F.lit(None)).otherwise(F.col("latitude")).alias("latitude"),
    F.when(F.col("longitude").isin(0.0, 0), F.lit(None)).otherwise(F.col("longitude")).alias("longitude"),
    "dicofre", "tipo_entidade", "tipo_instalacao", "estrutura_producao",
    "freguesia", "concelho", "distrito", "svl", "dsavr",
    
    F.when(F.col("is_nova"), F.col("tipo_exploracao")).otherwise(F.col("des_tipo_exploracao")).alias("tipo_exploracao_codigo"),
    F.col("des_sistema_exploracao_origem").alias("sistema_exploracao"),
    F.col("des_cn_valor").alias("cabecas_normais"),
    
    F.coalesce(F.col("des_nr_des_agrupados"), F.lit(0)).alias("nr_des_agrupadas"),
    
    F.when(F.col("is_nova"), F.coalesce(F.col("data_inicio_atividade"), F.to_date(F.lit("1900-01-01"))))
     .when(F.col("is_velha") & F.col("des_is_first_record") & F.col("data_inicio_atividade").isNotNull() & (F.col("data_inicio_atividade") < F.col("des_data_inicio")), F.col("data_inicio_atividade"))
     .otherwise(F.col("des_data_inicio")).alias("data_inicio"),
     
    F.when(F.col("is_nova"), F.to_date(F.lit("9999-12-31")))
     .otherwise(F.coalesce(F.col("des_data_fim"), F.to_date(F.lit("9999-12-31")))).alias("data_fim"),
     
    # CORREÇÃO DA RASTREABILIDADE - Em vez de sobrescrever, concatena ambos mantendo os mesmos nomes.
    # O concat_ws ignora automaticamente os campos que estiverem nulos
    F.concat_ws(" | ", F.col("des_meta_source_file"), F.col("meta_source_file")).alias("meta_source_file"),
    F.concat_ws(" | ", F.col("des_meta_source_file_date").cast("string"), F.col("meta_source_file_date").cast("string")).alias("meta_source_file_date")
)


# 7. TRADUÇÃO DE CÓDIGOS E SPLIT (CICLO/DIMENSÃO)
df_gold = df_gold.withColumn(
    "tipo_exploracao_extenso",
    F.when(F.col("tipo_exploracao_codigo").isNull() | (F.col("tipo_exploracao_codigo") == ""), F.lit(None))
     .when(F.col("tipo_exploracao_codigo") == 'IC', "Industrial - Ciclo Completo")
     .when(F.col("tipo_exploracao_codigo") == 'FC', "Familiar - Ciclo Completo")
     .when(F.col("tipo_exploracao_codigo") == 'CC', "Caseiro - Ciclo Completo")
     .when(F.col("tipo_exploracao_codigo") == 'IA', "Industrial - Recria e Acabamento")
     .when(F.col("tipo_exploracao_codigo") == 'FA', "Familiar - Recria e Acabamento")
     .when(F.col("tipo_exploracao_codigo") == 'CA', "Caseiro - Recria e Acabamento")
     .when(F.col("tipo_exploracao_codigo") == 'IP', "Industrial de Cria - Produção de Leitões")
     .when(F.col("tipo_exploracao_codigo") == 'FP', "Familiar de Cria - Produção de Leitões")
     .when(F.col("tipo_exploracao_codigo") == 'CP', "Caseiro de Cria - Produção de Leitões")
     .when(F.col("tipo_exploracao_codigo") == 'SM', "Sem Movimento")
     .when(F.col("tipo_exploracao_codigo") == 'AT', "Atípica")
     .otherwise(F.lit(None))
).withColumn(
    "tipo_exploracao_ciclo", 
    F.when(F.col("tipo_exploracao_extenso").contains(" - "), F.split(F.col("tipo_exploracao_extenso"), " - ").getItem(0)).otherwise(F.lit(None))
).withColumn(
    "tipo_exploracao_dimensao", 
    F.when(F.col("tipo_exploracao_extenso").contains(" - "), F.split(F.col("tipo_exploracao_extenso"), " - ").getItem(1)).otherwise(F.lit(None))
)


# 8. ESTADO ATUAL E FORMATATAÇÃO FINAL
df_gold = df_gold.withColumn("estado", F.when(F.col("data_fim") == F.to_date(F.lit("9999-12-31")), "Ativa").otherwise("Inativa"))

df_final_hist = df_gold.withColumn("NK_Marca", F.concat(F.lit("NK_"), F.col("marca"))) \
                       .withColumn("SK_Marca_Historico", F.concat(F.lit("SK_"), F.col("marca"), F.lit("_"), F.date_format(F.col("data_inicio"), "yyyyMMdd")))

# Substituir Nulos por "Não Definido"
string_cols = ["dicofre", "tipo_entidade", "tipo_instalacao", "estrutura_producao", 
               "tipo_exploracao_codigo", "sistema_exploracao", "tipo_exploracao_extenso", 
               "tipo_exploracao_ciclo", "tipo_exploracao_dimensao", 
               "freguesia", "concelho", "distrito", "svl", "dsavr"]

for c in string_cols:
    df_final_hist = df_final_hist.withColumn(c, F.coalesce(F.col(c), F.lit("Não Definido")))


# 9. CRIAÇÃO DA TABELA DE DIMENSÃO GEOGRÁFICA (Sem duplicação)
# CORREÇÃO DA DUPLICAÇÃO: Agrupa estritamente pelos campos da geografia, garantindo a unicidade.
# Os ficheiros correspondentes são fundidos numa string (ex: "file_A.csv | file_B.csv") 
df_dim_geografia = df_final_hist.groupBy(
    "SK_Geo_Exp", 
    "freguesia", 
    "concelho", 
    "distrito", 
    "svl", 
    "dsavr"
).agg(
    F.concat_ws(" | ", F.collect_set("meta_source_file")).alias("meta_source_file"),
    F.max("meta_source_file_date").alias("meta_source_file_date"),
    F.current_timestamp().alias("audit_gold_refresh_timestamp")
)


# 10. SELEÇÃO FINAL DA DIMENSÃO HISTÓRICA
df_final_hist = df_final_hist.select(
    "SK_Marca_Historico", "NK_Marca", "SK_Geo_Exp", "marca", 
    "data_inicio", "data_fim", "estado",
    "tipo_entidade", "tipo_instalacao", "estrutura_producao", "sistema_exploracao",
    "tipo_exploracao_codigo", "tipo_exploracao_extenso", "tipo_exploracao_ciclo", "tipo_exploracao_dimensao",
    "cabecas_normais", "dicofre", "latitude", "longitude",
    "nr_des_agrupadas",
    "meta_source_file", "meta_source_file_date", 
    F.current_timestamp().alias("audit_gold_refresh_timestamp")
)

# 11. GUARDAR NA GOLD
spark.sql("CREATE SCHEMA IF NOT EXISTS gld")

df_final_hist.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gld.dim_exploracao_scd")
df_dim_geografia.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gld.dim_exp_geografia")

print(f"Sucesso! Dimensões 'gld.dim_exploracao_scd' e 'gld.dim_exp_geografia' geradas.")

StatementMeta(, 04314e6f-6d50-4e2b-b4b5-051901dd44ad, 8, Finished, Available, Finished, False)

CONSTRUÇÃO DA DIMENSÃO: gld.dim_exploracao_historico (SCD TIPO 2 - COMPRIMIDA)

Sucesso! Dimensões 'gld.dim_exploracao_scd' e 'gld.dim_exp_geografia' geradas.


## Criaçao de tabela de auditoria do pipeline

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F



# VALIDAR RUN ID RECEBIDO DO PIPELINE

if run_id is None or str(run_id).strip() == "":
    raise ValueError(
        "O parâmetro 'run_id' não foi recebido do pipeline. "
        "Confirma se a atividade Notebook está configurada com "
        "run_id = @pipeline().parameters.p_run_id"
    )

run_id = str(run_id).strip()

print(f"Run ID recebido do pipeline: {run_id}")



# COMPARAR A VERSÃO ATUAL COM A VERSÃO DELTA ANTERIOR

def calcular_metricas_delta(
    table_name,
    chaves,
    colunas_ignorar=None
):
    colunas_ignorar = colunas_ignorar or []

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"A tabela '{table_name}' não existe."
        )

    delta_table = DeltaTable.forName(
        spark,
        table_name
    )

    # Apenas as duas versões mais recentes são necessárias
    historico = (
        delta_table.history(2)
        .select(
            "version",
            "timestamp",
            "operation"
        )
        .orderBy(
            F.col("version").desc()
        )
    )

    versoes = [
        row["version"]
        for row in historico
        .select("version")
        .collect()
    ]

    if not versoes:
        raise ValueError(
            f"Não foi encontrado histórico Delta para "
            f"'{table_name}'."
        )

    versao_atual = versoes[0]

    df_atual = (
        spark.read
        .format("delta")
        .option(
            "versionAsOf",
            versao_atual
        )
        .table(table_name)
    )

    total_rows = df_atual.count()

    # Primeira execução da tabela
    if len(versoes) == 1:
        return {
            "current_version": versao_atual,
            "previous_version": None,
            "total_rows": total_rows,
            "rows_added": total_rows,
            "rows_updated": 0,
            "rows_deleted": 0
        }

    versao_anterior = versoes[1]

    df_anterior = (
        spark.read
        .format("delta")
        .option(
            "versionAsOf",
            versao_anterior
        )
        .table(table_name)
    )

    # Validar chaves
    for chave in chaves:
        if chave not in df_atual.columns:
            raise ValueError(
                f"A chave '{chave}' não existe na versão atual "
                f"da tabela '{table_name}'."
            )

        if chave not in df_anterior.columns:
            raise ValueError(
                f"A chave '{chave}' não existe na versão anterior "
                f"da tabela '{table_name}'."
            )

    chaves_atuais = (
        df_atual
        .select(*chaves)
        .distinct()
    )

    chaves_anteriores = (
        df_anterior
        .select(*chaves)
        .distinct()
    )

    # Linhas adicionadas
    rows_added = (
        chaves_atuais
        .join(
            chaves_anteriores,
            on=chaves,
            how="left_anti"
        )
        .count()
    )

    # Linhas eliminadas
    rows_deleted = (
        chaves_anteriores
        .join(
            chaves_atuais,
            on=chaves,
            how="left_anti"
        )
        .count()
    )

    # Colunas a comparar para atualizações reais
    colunas_comparacao = [
        coluna
        for coluna in df_atual.columns
        if coluna in df_anterior.columns
        and coluna not in chaves
        and coluna not in colunas_ignorar
    ]

    atual = df_atual.alias("atual")
    anterior = df_anterior.alias("anterior")

    condicao_join = F.lit(True)

    for chave in chaves:
        condicao_join = (
            condicao_join
            & F.col(f"atual.{chave}")
            .eqNullSafe(
                F.col(f"anterior.{chave}")
            )
        )

    condicao_alteracao = F.lit(False)

    for coluna in colunas_comparacao:
        condicao_alteracao = (
            condicao_alteracao
            | ~F.col(f"atual.{coluna}")
            .eqNullSafe(
                F.col(f"anterior.{coluna}")
            )
        )

    rows_updated = (
        atual
        .join(
            anterior,
            on=condicao_join,
            how="inner"
        )
        .filter(condicao_alteracao)
        .select(
            *[
                F.col(f"atual.{chave}").alias(chave)
                for chave in chaves
            ]
        )
        .distinct()
        .count()
    )

    return {
        "current_version": versao_atual,
        "previous_version": versao_anterior,
        "total_rows": total_rows,
        "rows_added": rows_added,
        "rows_updated": rows_updated,
        "rows_deleted": rows_deleted
    }



# ESCREVER UMA LINHA NA TABELA DE AUDITORIA

def escrever_auditoria(
    layer,
    notebook_name,
    table_name,
    metricas,
    run_id,
    status="success"
):
    if run_id is None or str(run_id).strip() == "":
        raise ValueError(
            f"Não é possível auditar '{table_name}': "
            "o run_id está vazio."
        )

    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS audit"
    )

    audit_df = (
        spark.createDataFrame([
            Row(
                run_id=str(run_id).strip(),
                layer=str(layer),
                notebook_name=str(notebook_name),
                table_name=str(table_name),
                status=str(status),
                total_rows=int(metricas["total_rows"]),
                rows_added=int(metricas["rows_added"]),
                rows_updated=int(metricas["rows_updated"]),
                rows_deleted=int(metricas["rows_deleted"])
            )
        ])
        .withColumn(
            "audit_timestamp",
            F.current_timestamp()
        )
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(
            "audit.pipeline_audit_log"
        )
    )

    print(
        f"Auditoria registada | "
        f"run_id={run_id} | "
        f"table={table_name} | "
        f"versão anterior={metricas['previous_version']} | "
        f"versão atual={metricas['current_version']} | "
        f"total={metricas['total_rows']} | "
        f"added={metricas['rows_added']} | "
        f"updated={metricas['rows_updated']} | "
        f"deleted={metricas['rows_deleted']}"
    )



# CALCULAR MÉTRICAS DAS DUAS TABELAS

metricas_hist = calcular_metricas_delta(
    table_name="gld.dim_exploracao_scd",
    chaves=[
        "SK_Marca_Historico"
    ],
    colunas_ignorar=[
        "audit_gold_refresh_timestamp"
    ]
)

metricas_geo = calcular_metricas_delta(
    table_name="gld.dim_exp_geografia",
    chaves=[
        "SK_Geo_Exp"
    ],
    colunas_ignorar=[
        "audit_gold_refresh_timestamp"
    ]
)

print("Métricas dim_exploracao_scd:")
print(metricas_hist)

print("Métricas dim_exp_geografia:")
print(metricas_geo)



# ESCREVER NA AUDITORIA COM O RUN ID DO PIPELINE

escrever_auditoria(
    layer="gold",
    notebook_name="dim_exploracao_scd",
    table_name="gld.dim_exploracao_scd",
    metricas=metricas_hist,
    run_id=run_id
)

escrever_auditoria(
    layer="gold",
    notebook_name="dim_exp_geografia",
    table_name="gld.dim_exp_geografia",
    metricas=metricas_geo,
    run_id=run_id
)

print(
    f"Auditoria concluída com sucesso. "
    f"Run ID: {run_id}"
)

## Validação 

In [ ]:
# # BLOCO DE VALIDAÇÃO: MAPEAMENTO GEOGRÁFICO (CORRIGIDO)
# print(f"\n{'*'*20} RELATÓRIO DE VALIDAÇÃO GEOGRÁFICA {'*'*20}")

# # 1. Adicionar a lista do algarve à soma das listas
# todas_as_listas = list_acores + list_madeira + list_alentejo + list_centro + list_lvt + list_norte + list_algarve

# # Filtrar o que não está nas listas E que não seja Nulo (para não poluir o relatório)
# df_unmapped = df_joined.filter(
#     (~F.col("svl").isin(todas_as_listas)) & 
#     (F.col("svl").isNotNull())
# ).select("svl").distinct()

# count_unmapped = df_unmapped.count()

# if count_unmapped > 0:
#     print(f"ALERTA: Foram encontrados {count_unmapped} valores de SVL que NÃO estão nas listas de mapeamento:")
#     df_unmapped.show(truncate=False)
# else:
#     print("SUCESSO: Todos os valores de SVL (não nulos) foram mapeados corretamente.")

# # 2. Resumo da distribuição por DSAVR
# print("\nDistribuição por DSAVR (Região):")
# df_joined.groupBy("dsavr").count().orderBy("dsavr").show()

# print(f"{'*'*75}\n")

StatementMeta(, 7f13116d-28a4-4808-9639-85b6be44a5a2, 9, Finished, Available, Finished, False)


******************** RELATÓRIO DE VALIDAÇÃO GEOGRÁFICA ********************


SUCESSO: Todos os valores de SVL (não nulos) foram mapeados corretamente.

Distribuição por DSAVR (Região):


+--------+-----+
|   dsavr|count|
+--------+-----+
|    NULL| 1216|
|Alentejo|32661|
| Algarve| 2918|
|  Açores| 4210|
|  Centro|63629|
|     LVT|30088|
| Madeira|  885|
|   Norte|22335|
+--------+-----+

***************************************************************************



In [6]:
# from pyspark.sql import functions as F

# tabela = "gld.dim_exploracao_scd"
# chave = "SK_Marca_Historico"

# df = spark.table(tabela)

# # Resumo
# total_linhas = df.count()
# total_chaves_distintas = df.select(chave).distinct().count()
# linhas_repetidas = total_linhas - total_chaves_distintas

# print(f"Tabela: {tabela}")
# print(f"Total de linhas: {total_linhas}")
# print(f"Chaves distintas: {total_chaves_distintas}")
# print(f"Linhas repetidas além da primeira: {linhas_repetidas}")

# # Mostrar as chaves repetidas
# duplicados = (
#     df.groupBy(chave)
#       .agg(F.count("*").alias("quantidade"))
#       .filter(F.col("quantidade") > 1)
#       .orderBy(F.col("quantidade").desc())
# )

# duplicados.show(truncate=False)


StatementMeta(, e40f9455-efc8-4974-ba2d-dbed7da2d1d5, 8, Finished, Available, Finished, False)

Tabela: gld.dim_exploracao_scd
Total de linhas: 157942
Chaves distintas: 157942
Linhas repetidas além da primeira: 0
+------------------+----------+
|SK_Marca_Historico|quantidade|
+------------------+----------+
+------------------+----------+

